In [51]:
import sys
import os
import requests
from datetime import datetime, timedelta
from fastapi import FastAPI, Query
from fastapi.testclient import TestClient

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from shared.config import Config
import yfinance as yf

In [52]:
data = yf.Ticker("AAPL").history(period="1mo")

In [53]:
data.tail(5)

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2026-04-02 00:00:00-04:00,254.199997,256.130005,250.649994,255.919998,31289400,0.0,0.0
2026-04-06 00:00:00-04:00,256.510010,262.160004,256.459991,258.859985,29329900,0.0,0.0
2026-04-07 00:00:00-04:00,256.160004,256.200012,245.699997,253.500000,62148000,0.0,0.0
2026-04-08 00:00:00-04:00,258.450012,259.750000,256.529999,258.899994,41032800,0.0,0.0
2026-04-09 00:00:00-04:00,259.000000,261.119995,256.070007,260.489990,28084500,0.0,0.0


In [54]:
data["Open"].iloc[-1]

np.float64(259.0)

In [55]:
stock = yf.Ticker("AAPL")
data = stock.history(start="2024-01-01", end="2024-03-01")
data.tail(1)

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2024-02-29 00:00:00-05:00,179.635172,180.923451,177.910859,179.119858,136682600,0.0,0.0


In [56]:
end = datetime.now()
start = end - timedelta(days=1)

data = yf.Ticker("AAPL").history(start=start, end=end)
data

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2026-04-09 00:00:00-04:00,259.0,261.119995,256.070007,260.48999,28084500,0.0,0.0


In [57]:
end = datetime.now()
start = end - timedelta(days=1)
def get_stocks(symbols:str,start:str,end:str,period:str):
    stock = yf.Ticker(symbols)
    if period == "scheduled":
        data = stock.history(end=end)
        data.tail(1)
    elif period == "range":
        data = stock.history(start=start, end=end)
    else:
        data = stock.history(period="1d")
    try:   
        return data.reset_index().to_dict(orient="records")
    except Exception as e:
        return {"error": str(e)}

In [58]:
app = FastAPI()

@app.get("/stocks")
def stocks_endpoint(
    symbols: str,
    start: str = Query(None, description="Date in YYYY-MM-DD format for historical data"),
    end: str = Query(None, description="Date in YYYY-MM-DD format for historical data"),
    period: str = Query("current", enum=["current", "scheduled", "range"]),
):
    return get_stocks(symbols,start,end,period)


client = TestClient(app)

In [59]:
res_current = client.get("/stocks", params={"symbols": "AAPL"})
print(res_current.json(), "\n")

[{'Date': '2026-04-09T00:00:00-04:00', 'Open': 259.0, 'High': 261.1199951171875, 'Low': 256.07000732421875, 'Close': 260.489990234375, 'Volume': 28084500, 'Dividends': 0.0, 'Stock Splits': 0.0}] 

